In [1]:
import pandas as pd


sensors = {}
for i in range(1, 36+1):
    dfSensor = pd.read_excel('../data/DraginoSoilMositure_Morocco_Season1.xlsx', sheet_name=f'Sensor {i}',
                             usecols=[0, 1])
    dfSensor.rename(columns={"Row Labels": "datetime", "Average of water_SOIL": "sm_value"}, inplace=True)
    dfSensor = dfSensor.set_index("datetime")
    full_index = pd.date_range(dfSensor.index.min(), dfSensor.index.max(), freq="h")
    dfSensor = dfSensor.reindex(full_index)
    dfSensor.reset_index(inplace=True, names='datetime')
    dfSensor["sm_value"] = dfSensor["sm_value"].interpolate()
    sensors[i] = dfSensor

dfMeteo = pd.read_csv('../data/open-meteo.csv')
dfMeteo["datetime"] = pd.to_datetime(dfMeteo["datetime"])
dfMeteo = dfMeteo.set_index("datetime")
dfMeteo = dfMeteo.reindex(full_index)
dfMeteo.reset_index(inplace=True, names='datetime')

dfMeteo_F = pd.read_csv('../data/open-meteo.csv')
dfMeteo_F["datetime"] = pd.to_datetime(dfMeteo_F["datetime"])
dfMeteo_F = dfMeteo_F.set_index("datetime")
dfMeteo_F = dfMeteo_F.reindex(full_index)
dfMeteo_F.reset_index(inplace=True, names='datetime')



dfLAI = pd.read_excel('../data/LAI_Morocco_Season1.xlsx', sheet_name=f'Sheet1')
dfLAI["datetime"] = pd.to_datetime(dfLAI["datetime"]) + pd.to_timedelta("12:00:00")
dfLAI = dfLAI.set_index("datetime")
# full_index1 = pd.date_range(dfLAI.index.min(), dfLAI.index.max(), freq="h")
full_index2 = pd.date_range(start='2024-12-12  12:00:00', end='2025-05-15 13:00:00', freq="h")
dfLAI = dfLAI.reindex(full_index)

values = {
    "Tititcaca-CROP": 0,
    "Tititcaca-Sensor": 0,
    "Tititcaca-Farmer": 0,
    "ICBA-CROP": 0,
    "ICBA-Sensor": 0,
    "ICBA-Farmer": 0
}
dfLAI.loc["2024-12-12 12:00:00"] = values

dfLAI[['Tititcaca-CROP', 'Tititcaca-Sensor', 'Tititcaca-Farmer', 'ICBA-CROP',
       'ICBA-Sensor', 'ICBA-Farmer']] = dfLAI[['Tititcaca-CROP', 'Tititcaca-Sensor', 'Tititcaca-Farmer', 'ICBA-CROP',
                                               'ICBA-Sensor', 'ICBA-Farmer']].interpolate()

mapToSensor = {'Tititcaca-CROP': [5, 6, 23, 24, 31, 32],
               'Tititcaca-Sensor': [3, 4, 19, 20, 35, 36],
               'Tititcaca-Farmer': [9, 10, 13, 14, 29, 30],
               'ICBA-CROP': [1, 2, 15, 16, 33, 34],
               'ICBA-Sensor': [7, 8, 21, 22, 25, 26],
               'ICBA-Farmer': [11, 12, 17, 18, 27, 28]}

for i in range(1, 37):
    for k, v in mapToSensor.items():
        if i in v:
            dfLAI[f'{i}'] = dfLAI[k]

dfLAI.drop(['Tititcaca-CROP', 'Tititcaca-Sensor', 'Tititcaca-Farmer', 'ICBA-CROP',
            'ICBA-Sensor', 'ICBA-Farmer'], axis=1)
dfLAI.reset_index(inplace=True, names='datetime')


dfMeteoStation = pd.read_excel('../data/WeatherStation_Slimania_Season1.xlsx', usecols=[1,2,3,4,5,6,7,8,9])
dfMeteoStation["datetime"] = dfMeteoStation["datetime"].dt.round('h')
dfMeteoStation_rev = dfMeteoStation.groupby('datetime').mean()

#dfMeteoStation_rev = dfMeteoStation_rev.set_index("datetime")
dfMeteoStation_rev = dfMeteoStation_rev.reindex(full_index)
values = {
    "Temp": 0,
    "Hum": 0,
    "Int": 0,
    "UVI": 0,
    "WS": 0,
    "WD": 0,
    "RG": 0,
    "BP": 0
}
dfMeteoStation_rev.loc["2024-12-12 12:00:00"] = values
dfMeteoStation_rev[['Temp', 'Hum', 'Int', 'UVI', 'WS', 'WD', 'RG', 'BP']] = dfMeteoStation_rev[['Temp', 'Hum', 'Int', 'UVI', 'WS', 'WD', 'RG', 'BP']].interpolate()
dfMeteoStation_rev.reset_index(inplace=True, names='datetime')

KeyError: "None of ['datetime'] are in the columns"

# Irrigation

In [244]:
full_index2 = pd.date_range(start='2024-12-12  12:00:00', end='2025-05-15 13:00:00', freq="h")

dfIRR = pd.DataFrame(columns=['f1','f2','f3','f4','f5','f6','f7','f8','f9','f10','f11','f12','f13','f14','f15','f16','f17','f18'], index=full_index2)

#dfIRR = dfIRR.reindex(full_index)
#dfIRR.reset_index(inplace=True, names='datetime')
dfIRR.fillna(0.0, inplace=True)

/var/folders/66/rpqcrym93h90yphhq7kgwhb00000gn/T/ipykernel_12029/2691280908.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dfIRR.fillna(0.0, inplace=True)


In [245]:
dfIRR.loc[pd.to_datetime('2024-12-12 15:00:00'), 'f1'] = 99.2

In [246]:
dfIrrigation = pd.read_excel('../data/IrrigationEvents_Morocco_Season1.xlsx', sheet_name='Irrigation')
dfIrrigation.rename(columns={"Date": "datetime", "Irrigation Duration": "duration", "Irrigation Volume": "volume"},
                     inplace=True)
dfIrrigation.drop(['Soil Mositure before irrigation (only Sensor)','duration'], axis=1, inplace=True)
dfIrrigation["datetime"] = pd.to_datetime(dfIrrigation["datetime"]) + pd.to_timedelta("12:00:00")
dfIrrigation.sort_values("datetime", ascending=True, inplace=True)
CW = ['f1', 'f3', 'f8', 'f12', 'f16','f17']
F = ['f5', 'f6', 'f7', 'f9', 'f14', 'f15']
S = ['f2', 'f4', 'f10', 'f11', 'f13','f18']
# dfIrrigation.loc[dfIrrigation['Plot'] == 'CROPWAT', 'Plot'] = 0
dfIrrigation["Plot"] = dfIrrigation["Plot"].apply(
    lambda x: CW if x == "CROPWAT" else S if x == "Farmer" else F
)
print(dfIrrigation.columns)

Index(['datetime', 'Plot', 'volume'], dtype='object')


In [248]:
for irr_ev in dfIrrigation.iterrows():
    #print(irr_ev[1]['Plot'])
    for p in irr_ev[1]['Plot']:
        dfIRR.loc[pd.to_datetime(irr_ev[1]['datetime']), p] = irr_ev[1]['volume']
        #print(dfIRR.loc[irr_ev[1]['datetime'], p])
        #pd.to_datetime(irr_ev[1]['datetime']),

In [249]:
print(sensors[1].shape, sensors[1].columns)
print(dfIRR.shape, dfIRR.columns)
print(dfMeteo.shape, dfMeteo.columns)

(3698, 2) Index(['datetime', 'sm_value'], dtype='object')
(3698, 18) Index(['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11',
       'f12', 'f13', 'f14', 'f15', 'f16', 'f17', 'f18'],
      dtype='object')
(3698, 12) Index(['datetime', 'temperature_2m', 'relative_humidity_2m', 'cloud_cover',
       'wind_speed_10m', 'wind_direction_100m', 'soil_temperature_0_to_7cm',
       'soil_temperature_7_to_28cm', 'soil_temperature_28_to_100cm', 'rain',
       'precipitation', 'evapotranspiration'],
      dtype='object')


In [252]:
dfIRR.index

DatetimeIndex(['2024-12-12 12:00:00', '2024-12-12 13:00:00',
               '2024-12-12 14:00:00', '2024-12-12 15:00:00',
               '2024-12-12 16:00:00', '2024-12-12 17:00:00',
               '2024-12-12 18:00:00', '2024-12-12 19:00:00',
               '2024-12-12 20:00:00', '2024-12-12 21:00:00',
               ...
               '2025-05-15 04:00:00', '2025-05-15 05:00:00',
               '2025-05-15 06:00:00', '2025-05-15 07:00:00',
               '2025-05-15 08:00:00', '2025-05-15 09:00:00',
               '2025-05-15 10:00:00', '2025-05-15 11:00:00',
               '2025-05-15 12:00:00', '2025-05-15 13:00:00'],
              dtype='datetime64[ns]', length=3698, freq='h')

In [267]:
for i in range (1,36+1):
    sensors[i].set_index('datetime', inplace=True)

In [268]:
dfMeteo.set_index('datetime', inplace=True)

KeyError: "None of ['datetime'] are in the columns"

In [266]:
dfMeteo.index

DatetimeIndex(['2024-12-12 12:00:00', '2024-12-12 13:00:00',
               '2024-12-12 14:00:00', '2024-12-12 15:00:00',
               '2024-12-12 16:00:00', '2024-12-12 17:00:00',
               '2024-12-12 18:00:00', '2024-12-12 19:00:00',
               '2024-12-12 20:00:00', '2024-12-12 21:00:00',
               ...
               '2025-05-15 04:00:00', '2025-05-15 05:00:00',
               '2025-05-15 06:00:00', '2025-05-15 07:00:00',
               '2025-05-15 08:00:00', '2025-05-15 09:00:00',
               '2025-05-15 10:00:00', '2025-05-15 11:00:00',
               '2025-05-15 12:00:00', '2025-05-15 13:00:00'],
              dtype='datetime64[ns]', name='datetime', length=3698, freq=None)

In [279]:
full_index2 = pd.date_range(start='2024-12-12  12:00:00', end='2025-05-15 13:00:00', freq="h")
meteo_fields = ['temperature_2m', 'relative_humidity_2m', 'cloud_cover',
       'wind_speed_10m', 'wind_direction_100m', 'soil_temperature_0_to_7cm',
       'soil_temperature_7_to_28cm', 'soil_temperature_28_to_100cm', 'rain',
       'precipitation', 'evapotranspiration']
all_fields = []

for i, field in enumerate(['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11',
       'f12', 'f13', 'f14', 'f15', 'f16', 'f17', 'f18'], start=1):
    df_new = pd.DataFrame(columns=['s_a','s_b','irr','temperature_2m', 'relative_humidity_2m', 'cloud_cover',
       'wind_speed_10m', 'wind_direction_100m', 'soil_temperature_0_to_7cm',
       'soil_temperature_7_to_28cm', 'soil_temperature_28_to_100cm', 'rain',
       'precipitation', 'evapotranspiration'], index=full_index2)
    df_new['irr'] = dfIRR[field]
    df_new['s_a'] = sensors[i*2]['sm_value']
    df_new['s_b'] = sensors[i*2-1]['sm_value']
    df_new[meteo_fields] = dfMeteo[meteo_fields]
    df_new.to_csv(f'field_{field}',index_label='datetime')


In [285]:
df = pd.read_csv('field_f1')